# L4: Optimize DSPy Agent with DSPy Optimizer

In [0]:
%pip install mlflow>=3.0 databricks-agents databricks-feature-engineering --upgrade
dbutils.library.restartPython()

In [0]:
api_base = f'https://{spark.conf.get("spark.databricks.workspaceUrl")}/serving-endpoints'
api_key = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
user_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()

In [0]:
import mlflow

In [0]:
mlflow.__version__

In [0]:
mlflow.get_registry_uri()


In [0]:
mlflow.get_tracking_uri()

In [0]:
import os
EXP_NAME = f"/Workspace/Users/{user_name}/mlflow_artifacts_dspy/mlflow_dspy_lession4"
ARTIFACT_PATH = f"dbfs:/Volumes/sandbox_db/agentic_rag/mlflow_artifacts/dspy_lession4"

if mlflow.get_experiment_by_name(EXP_NAME) is None:
    mlflow.create_experiment(name=EXP_NAME, artifact_location=ARTIFACT_PATH)
mlflow.set_experiment(EXP_NAME)

In [0]:
mlflow.get_experiment_by_name(EXP_NAME)

In [0]:
mlflow.dspy.autolog(log_evals=True, log_compiles=True, log_traces_from_compile=True)

In [0]:
import dspy

lm = dspy.LM(
    "databricks/databricks-llama-4-maverick",
    api_key=api_key,
    api_base=api_base
)
dspy.configure(lm=lm)

## Build a RAG Agent

In [0]:
def search_wikipedia(query: str) -> list[str]:
    results = dspy.ColBERTv2(url="http://20.102.90.50:2017/wiki17_abstracts")(query, k=3)
    return [x["text"] for x in results]

react = dspy.ReAct("question -> answer", tools=[search_wikipedia])

In [0]:
import json

# Load trainset
trainset = []
with open("trainset.jsonl", "r") as f:
    for line in f:
        trainset.append(dspy.Example(**json.loads(line)).with_inputs("question"))

# Load valset
valset = []
with open("valset.jsonl", "r") as f:
    for line in f:
        valset.append(dspy.Example(**json.loads(line)).with_inputs("question"))

In [0]:
# Overview of the dataset.
print(trainset[0])

In [0]:
tp = dspy.MIPROv2(
    metric=dspy.evaluate.answer_exact_match,
    auto="light",
    num_threads=16
)

In [0]:
dspy.cache.load_memory_cache("./memory_cache.pkl")

In [0]:
optimized_react = tp.compile(
    react,
    trainset=trainset,
    valset=valset,
    requires_permission_to_run=False,
)

In [0]:
optimized_react.react.signature

In [0]:
optimized_react.react.demos

In [0]:
evaluator = dspy.Evaluate(
    metric=dspy.evaluate.answer_exact_match,
    devset=valset,
    display_table=True,
    display_progress=True,
    num_threads=24,
)

In [0]:
original_score = evaluator(react)
print(f"Original score: {original_score}")

In [0]:
optimized_score = evaluator(optimized_react)
print(f"Optimized score: {optimized_score}")